# 04 · 稳健组合推荐

**目标**：结合 `03` 的 WFA 结果 + 硬编码基线，输出最终稳健组合。

**输入**：
- `wfa_results.csv`、`weight_history.csv`、`robustness_score.csv`
- `universe.csv`
- `etf_portfolio/robust_baselines.py`（3 个硬编码基线）

**输出**：
- 路线 B 稳健组合（等权 / 风险平价）
- 路线 A 动态组合（最新一期权重）
- `final_baselines.json`（最终交付清单）

In [ ]:
# ============================================================
# cell 0: imports + 全局参数
# ============================================================
from jqdata import *            # 聚宽 magic
import sys, os, json, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import pandas as pd
import numpy as np

PROJ = Path('/Users/huhao/src/codesnip/python/ai/028-jukuan').resolve()
sys.path.insert(0, str(PROJ))

from etf_portfolio.data_loader import load_parquet
from etf_portfolio.robust_baselines import BASELINES, get_baseline_weights, list_baselines
from etf_portfolio.risk_parity import risk_parity
from etf_portfolio.metrics import full_metrics

OUTPUT_DIR = PROJ / 'etf_portfolio' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 全局参数
ROBUST_TOP_N = 8          # 路线 B 入选 ETF 数量
ROBUST_MIN_FREQ = 0.30    # 入选最低 freq 阈值
print('硬编码基线:', list_baselines())

In [ ]:
# ============================================================
# cell 1: 加载 WFA 输出 + 候选池
# ============================================================
prices = load_parquet(OUTPUT_DIR / 'prices_daily.parquet')
wfa_metrics = pd.read_csv(OUTPUT_DIR / 'wfa_results.csv', parse_dates=['date'])
weight_hist = pd.read_csv(OUTPUT_DIR / 'weight_history.csv', parse_dates=['date'])
# 关键：用纯 code index 的那份（robustness_score.csv 是展示用，带中文名）
robust_score = pd.read_csv(OUTPUT_DIR / 'robustness_score_codes.csv', index_col=0)
uni = pd.read_csv(OUTPUT_DIR / 'universe.csv', dtype={'code': str})
uni_lookup = dict(zip(uni['code'], uni['name']))
print(f'WFA metrics: {len(wfa_metrics)} 行')
print(f'Weight history: {len(weight_hist)} 行')
print(f'稳健度评分: {len(robust_score)} 只 ETF')

In [ ]:
# ============================================================
# cell 2: 路线 B — 从稳健度评分选 Top N
# ============================================================
robust_score_named = robust_score.copy()
robust_score_named['name'] = [uni_lookup.get(c, c) for c in robust_score_named.index]
top_n = robust_score_named.head(ROBUST_TOP_N)
print(f'路线 B 候选 Top {ROBUST_TOP_N}:')
top_n[['name', 'freq', 'avg_weight', 'stability', 'n_obj']]

In [ ]:
# ============================================================
# cell 3: 路线 B — 等权组合 + 风险平价组合
# ============================================================
robust_codes = list(top_n.index)
print('入选 ETF:', robust_codes)

# 用近 5 年日收益算风险平价权重
end = prices.index.max()
start = end - pd.DateOffset(months=60)
rb_prices = prices[robust_codes].loc[start:end].dropna(how="any")
rb_ret = rb_prices.pct_change().dropna()
rb_cov = rb_ret.cov() * 252

# 等权
w_equal = pd.Series(np.ones(len(robust_codes)) / len(robust_codes), index=robust_codes)
# 风险平价
w_rp_arr = risk_parity(rb_cov.values)
w_rp = pd.Series(w_rp_arr, index=robust_codes)

b_robust = pd.DataFrame({
    'name':         [uni_lookup.get(c, c) for c in robust_codes],
    'equal_weight': w_equal.round(4),
    'rp_weight':    w_rp.round(4),
})
b_robust

In [ ]:
# ============================================================
# cell 4: 路线 A — 最新一期权重（动态调仓版）
# ============================================================
latest_weights = weight_hist.sort_values('date').groupby('objective').tail(1).set_index('objective')
latest_weights = latest_weights.drop(columns=['date'])
# 仅显示非零权重
def show_nonzero(row):
    nz = row[row > 0.001].sort_values(ascending=False)
    return pd.Series({uni_lookup.get(c, c): round(v, 4) for c, v in nz.items()})

print('路线 A 最新一期权重（按目标）：')
for obj in latest_weights.index:
    print(f'\n[{obj}]')
    print(show_nonzero(latest_weights.loc[obj]))

In [ ]:
# ============================================================
# cell 5: 汇总输出
# ============================================================
final_baselines = {}

# 1. 硬编码基线
for name, bl in BASELINES.items():
    final_baselines[name] = bl

# 2. 路线 B - 等权
final_baselines["稳健_等权_路线B"] = {
    "description": f"WFA 稳健度 Top {ROBUST_TOP_N} 等权组合",
    "holdings": [
        {"code": c, "name": uni_lookup.get(c, c), "weight": float(round(w, 4))}
        for c, w in w_equal.items()
    ],
}

# 3. 路线 B - 风险平价
final_baselines["稳健_风险平价_路线B"] = {
    "description": f"WFA 稳健度 Top {ROBUST_TOP_N} 风险平价组合",
    "holdings": [
        {"code": c, "name": uni_lookup.get(c, c), "weight": float(round(w, 4))}
        for c, w in w_rp.items()
    ],
}

# 写入 JSON
out_json = OUTPUT_DIR / 'final_baselines.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(final_baselines, f, ensure_ascii=False, indent=2)
print(f'已写入 {out_json}')
print(f'\n共 {len(final_baselines)} 个组合：')
for name in final_baselines:
    print(f'  - {name}')

In [ ]:
# ============================================================
# cell 6: 各组合日收益回测（用 60+ 月测试期）
# ============================================================
from etf_portfolio.data_loader import fetch_prices

# 统一测试期：取所有候选 ETF 上市后的最大共同区间
test_start = prices.index.min()
test_end = prices.index.max()
test_prices = prices.loc[test_start:test_end]
test_ret = test_prices.pct_change().dropna()

# 对每个组合构造日收益
port_ret_dict = {}
for name, bl in final_baselines.items():
    weights = {h["code"]: h["weight"] for h in bl["holdings"]}
    # 用全期测试
    common = test_ret.dropna(subset=list(weights.keys()), how='any')
    port_ret = sum(weights[c] * common[c] for c in weights)
    port_ret_dict[name] = port_ret.dropna()

print(f'各组合样本量: {[(k, len(v)) for k, v in port_ret_dict.items()]}')

In [ ]:
# ============================================================
# cell 7: 评估所有组合（年化收益 / 夏普 / 卡玛 / MDD）
# ============================================================
rows = []
for name, ret in port_ret_dict.items():
    m = full_metrics(ret)
    rows.append({
        '组合': name,
        '年化收益': m['annual_return'],
        '年化波动': m['annual_vol'],
        '夏普':     m['sharpe'],
        '卡玛':     m['calmar'],
        '最大回撤': m['max_drawdown'],
    })
summary_df = pd.DataFrame(rows).sort_values('夏普', ascending=False).reset_index(drop=True)
summary_df = summary_df.round(4)
summary_df

In [ ]:
summary_df.to_csv(OUTPUT_DIR / 'final_summary.csv', index=False, encoding='utf-8-sig')
print(f'已写入 final_summary.csv')

## 中间结论

- 路线 B：WFA 稳健度 Top 8 ETF + 等权 / 风险平价两种落地
- 路线 A：3 个目标的最新一期权重
- 硬编码基线 3 个（60/40、全天候、红利+海外+黄金）作为对照与应急
- 所有 8 个组合的指标对比见 `final_summary.csv`
- 进入 `05_报告与对比.ipynb` 出净值曲线 + 衰减率分布